# 623 Stride: threshold-free causal CNN only

This directory trains only the causal CNN. It receives lossless PC + cache-line-address bits, exactly the effective inputs read by `stride.cc`. Four stride-1 causal residual blocks use local kernel 7 and dilations 1/6/36/216, giving a contiguous 1,555-event sliding receptive field with exact 1,554-event chunk overlap and no future input. Captured Stride requests are supervision and comparator replay only; count argmax plus learned target ranking own the complete 64-offset action space, with no probability cutoff, request budget, degree cap, future-use threshold, or comparator gate.

In [ ]:
import hashlib, os, pathlib, shutil, subprocess, sys, tarfile, torch
from google.colab import userdata
assert torch.cuda.is_available(), 'Select a GPU runtime (A100 preferred)'
torch.set_float32_matmul_precision('high')
torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
REPO='/content/cache_arch'; TOKEN=userdata.get('GITHUB_TOKEN')
assert TOKEN, 'Add GITHUB_TOKEN to Colab Secrets'
ASKPASS='/content/cache_arch_git_askpass.sh'
pathlib.Path(ASKPASS).write_text('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token ;; *) echo "$GITHUB_TOKEN" ;; esac\n'); os.chmod(ASKPASS,0o700)
env=os.environ.copy(); env.update({'GIT_ASKPASS':ASKPASS,'GIT_TERMINAL_PROMPT':'0','GITHUB_TOKEN':TOKEN})
try:
    if not os.path.isdir(REPO): subprocess.run(['git','clone','https://github.com/Angelawoo572/cache_arch.git',REPO],check=True,env=env)
    else: subprocess.run(['git','-C',REPO,'pull','--ff-only','origin','main'],check=True,env=env)
finally: pathlib.Path(ASKPASS).unlink(missing_ok=True)
print(torch.cuda.get_device_name(0),subprocess.check_output(['git','-C',REPO,'rev-parse','HEAD'],text=True).strip())

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
RUN_ID='623_offline_cnn_stride_threshold_free_v7_seed7'; DRIVE_ROOT=f'/content/drive/MyDrive/cache_prefetch_623_stride_threshold_free/{RUN_ID}'
INPUT_DIR=f'{DRIVE_ROOT}/colab_input'; OUTPUT_ROOT=f'{DRIVE_ROOT}/colab_output'; os.makedirs(DRIVE_ROOT,exist_ok=True)
name=f'{RUN_ID}.colab_input.tar.gz'; uploaded=files.upload(); assert name in uploaded,f'Select {name}'
archive=f'{DRIVE_ROOT}/{name}'; pathlib.Path(archive).write_bytes(uploaded[name])
if os.path.isdir(INPUT_DIR): shutil.rmtree(INPUT_DIR)
os.makedirs(INPUT_DIR,exist_ok=True)
with tarfile.open(archive,'r:gz') as handle: handle.extractall(INPUT_DIR)
for record in pathlib.Path(f'{INPUT_DIR}/SHA256SUMS').read_text().splitlines():
    expected,item=record.split(maxsplit=1); item=item.lstrip('*')
    assert hashlib.sha256(pathlib.Path(f'{INPUT_DIR}/{item}').read_bytes()).hexdigest()==expected
print('verified',archive)

In [ ]:
import gzip, json
TRACE='623.xalancbmk_s-700B'; POLICY='stride'; ROLES=('train','guard','eval')
INPUTS={role:{'stream':f'{INPUT_DIR}/{TRACE}.{POLICY}.{role}_stream.csv.gz','candidates':f'{INPUT_DIR}/{TRACE}.{POLICY}.{role}_candidates.csv.gz'} for role in ROLES}
for role,items in INPUTS.items():
    for path in items.values(): assert os.path.isfile(path),path
manifest=json.loads(pathlib.Path(f'{INPUT_DIR}/collection_manifest.json').read_text())
expected={'status':'PASS','experiment_revision':'stride_threshold_free_split_v7','neural_role':'standalone_direct_action_prefetcher','source_decision_effective_external_input':['pc','addr'],'same_external_input_contract':True,'normal_policy_outputs_used_as_model_inputs':False,'normal_policy_candidates_used_as_model_inputs':False,'normal_policy_private_state_used_as_model_inputs':False,'normal_policy_outputs_used_as_training_targets':True,'normal_policy_request_rate_used_as_budget':False,'normal_policy_constants_used_by_neural_inference':False,'probability_threshold_used':False,'neural_degree_cap':None,'future_label_window_used':False,'inference_policy_hardcodes_used':False,'nn_generates_own_target_addresses':True}
bad={k:(manifest.get(k),v) for k,v in expected.items() if manifest.get(k)!=v}; assert not bad,bad
SCRIPT=f'{REPO}/formal_NN_training/experiments/623_offline_cnn_stride/python/train_and_offline_infer.py'

In [ ]:
LOCAL_OUTPUT=f'/content/{RUN_ID}_colab_output'
if os.path.isdir(LOCAL_OUTPUT): shutil.rmtree(LOCAL_OUTPUT)
os.makedirs(LOCAL_OUTPUT)
SPECS=[
 {'tag':'threshold_free_stride_cnn_c10','family':'cnn','size':10,'pair':'p0','parameters':5549},
 {'tag':'threshold_free_stride_cnn_c16','family':'cnn','size':16,'pair':'p1','parameters':11489},
 {'tag':'threshold_free_stride_cnn_c25','family':'cnn','size':25,'pair':'p2','parameters':24179},
]
SWEEP=[]
for spec in SPECS:
 out=f"{LOCAL_OUTPUT}/{spec['tag']}"; cmd=[sys.executable,SCRIPT,'--policy',POLICY]
 for role in ROLES: cmd += [f'--{role}-stream',INPUTS[role]['stream'],f'--{role}-candidates',INPUTS[role]['candidates']]
 cmd += ['--out-dir',out,'--model-family',spec['family'],'--model-size',str(spec['size']),'--pair-id',spec['pair'],'--device','cuda','--seed','7','--epochs','8','--chunk-len','1024','--accumulate-chunks','16']
 print('\nTraining',spec['tag'],' '.join(cmd),flush=True); subprocess.run(cmd,check=True)
 meta=json.loads(pathlib.Path(f'{out}/run_metadata.json').read_text())
 expected={'model_tag':spec['tag'],'parameter_count':spec['parameters'],'model_family':'cnn','track_model_family':'cnn','training_state_mode':'causal_dilated_tcn_over_chronological_stream','inference_history_mode':'chronological_sliding_context_with_exact_1554_event_overlap','cnn_temporal_layers':4,'cnn_kernel_size':7,'cnn_dilations':[1,6,36,216],'cnn_receptive_field_events':1555,'training_left_context_overlap':1554,'matched_normal_prefetcher':POLICY,'neural_role':'standalone_direct_action_prefetcher','same_external_input_contract':True,'normal_policy_outputs_used_as_model_inputs':False,'normal_policy_candidates_used_as_model_inputs':False,'normal_policy_private_state_used_as_model_inputs':False,'normal_policy_outputs_used_as_training_targets':True,'normal_policy_request_rate_used_as_budget':False,'normal_policy_constants_used_by_neural_inference':False,'probability_threshold_used':False,'neural_degree_cap':None,'future_label_window_used':False,'handcrafted_semantic_features_used':False,'manual_loss_weights_used':False,'training_regularization_used':False,'inference_policy_hardcodes_used':False,'learned_request_count':True,'nn_generates_own_target_addresses':True,'experiment_revision':'stride_threshold_free_split_v7'}
 bad={k:(meta.get(k),v) for k,v in expected.items() if meta.get(k)!=v}; assert not bad,bad
 SWEEP.append({k:meta[k] for k in ('model_tag','model_family','model_size','architecture_pair_id','parameter_count','decision_rule','offline_normal_entries','offline_nn_entries','heldout_behavior_metrics')})
if os.path.isdir(OUTPUT_ROOT): shutil.rmtree(OUTPUT_ROOT)
shutil.copytree(LOCAL_OUTPUT,OUTPUT_ROOT)
pathlib.Path(f'{OUTPUT_ROOT}/sweep_manifest.json').write_text(json.dumps({'trace':TRACE,'revision':'stride_threshold_free_split_v7','points':SWEEP},indent=2)+'\n')
print(json.dumps(SWEEP,indent=2))

In [ ]:
OUTPUT_ARCHIVE=f'{DRIVE_ROOT}/{RUN_ID}.colab_output.tar.gz'
with tarfile.open(OUTPUT_ARCHIVE,'w:gz') as archive:
 for item in pathlib.Path(OUTPUT_ROOT).iterdir(): archive.add(item,arcname=item.name)
print('DONE',OUTPUT_ARCHIVE,os.path.getsize(OUTPUT_ARCHIVE),'bytes')

Copy the output archive to the matching server run and launch replay. Older threshold/budget/future-window outputs are intentionally rejected by the server metadata checks.